# 📊 RAG Evaluation & Metrics

This notebook demonstrates pgVectorDB's built-in **RAG evaluator** for measuring retrieval quality.

### Metrics Computed
| Metric | What It Measures |
|:-------|:-----------------|
| **Precision@K** | Fraction of retrieved docs that are relevant |
| **Recall@K** | Fraction of all relevant docs that were retrieved |
| **F1@K** | Harmonic mean of precision and recall |
| **MAP** | Mean Average Precision (rank-aware) |
| **MRR** | Mean Reciprocal Rank (position of first relevant doc) |
| **NDCG@K** | Normalized Discounted Cumulative Gain |
| **Hit Rate** | Fraction of queries with ≥1 relevant result |

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from pgvectordb import RAGEvaluator

## 1. Basic Evaluation

Evaluate retrieval results against ground truth.

In [2]:
evaluator = RAGEvaluator(k=5)

# Simulated data
queries = [
    "What is vector search?",
    "How does BM25 work?",
    "Best database for AI?",
]

# What your system actually returned (doc IDs)
retrieved = [
    ["doc_1", "doc_3", "doc_7", "doc_2", "doc_5"],
    ["doc_4", "doc_6", "doc_1", "doc_8", "doc_2"],
    ["doc_2", "doc_1", "doc_9", "doc_3", "doc_4"],
]

# What SHOULD have been returned (ground truth)
ground_truth = [
    ["doc_1", "doc_2", "doc_3"],  # 3 relevant docs
    ["doc_4", "doc_6"],  # 2 relevant docs
    ["doc_2", "doc_9", "doc_10"],  # 3 relevant docs
]

result = evaluator.evaluate(queries, retrieved, ground_truth)
print(result)


Retrieval Evaluation Results:
Precision@K:  0.4667  (relevant in top K / K)
Recall@K:     0.8889  (relevant in top K / total relevant)
F1@K:         0.6120  (harmonic mean of P@K and R@K)
MAP@K:        0.8241  (mean average precision)
MRR:          1.0000  (1 / rank of first relevant)
NDCG@K:       0.8905  (ranking quality with discount)
Hit Rate@K:   1.0000  (queries with ≥1 relevant)



## 2. Per-Query Drill-Down

In [3]:
for i, query in enumerate(queries):
    metrics = evaluator.evaluate_single_query(
        retrieved_docs=retrieved[i],
        relevant_docs=ground_truth[i],
    )
    print(f"\n📝 Query: '{query}'")
    print(f"   Retrieved:    {retrieved[i]}")
    print(f"   Ground truth: {ground_truth[i]}")
    for metric, value in metrics.items():
        print(f"   {metric}: {value:.3f}")


📝 Query: 'What is vector search?'
   Retrieved:    ['doc_1', 'doc_3', 'doc_7', 'doc_2', 'doc_5']
   Ground truth: ['doc_1', 'doc_2', 'doc_3']
   precision: 0.600
   recall: 1.000
   f1_score: 0.750
   average_precision: 0.917
   reciprocal_rank: 1.000
   ndcg: 0.967
   hit: 1.000

📝 Query: 'How does BM25 work?'
   Retrieved:    ['doc_4', 'doc_6', 'doc_1', 'doc_8', 'doc_2']
   Ground truth: ['doc_4', 'doc_6']
   precision: 0.400
   recall: 1.000
   f1_score: 0.571
   average_precision: 1.000
   reciprocal_rank: 1.000
   ndcg: 1.000
   hit: 1.000

📝 Query: 'Best database for AI?'
   Retrieved:    ['doc_2', 'doc_1', 'doc_9', 'doc_3', 'doc_4']
   Ground truth: ['doc_2', 'doc_9', 'doc_10']
   precision: 0.400
   recall: 0.667
   f1_score: 0.500
   average_precision: 0.556
   reciprocal_rank: 1.000
   ndcg: 0.704
   hit: 1.000


## 3. Export Results

In [4]:
result_dict = result.to_dict()
print("📋 Exportable dict:")
for k, v in result_dict.items():
    print(f"  {k}: {v:.4f}")

📋 Exportable dict:
  precision: 0.4667
  recall: 0.8889
  f1_score: 0.6120
  map: 0.8241
  mrr: 1.0000
  ndcg: 0.8905
  hit_rate: 1.0000


## 4. Comparing K Values

See how metrics change at different K values.

In [5]:
print(f"{'K':>3} | {'Precision':>10} | {'Recall':>8} | {'F1':>6} | {'MRR':>6} | {'NDCG':>6}")
print("-" * 55)

for k_val in [1, 2, 3, 5, 10]:
    ev = RAGEvaluator(k=k_val)
    # Pad retrieved lists to at least k_val
    padded = [r + [f"pad_{j}" for j in range(k_val)] for r in retrieved]
    res = ev.evaluate(queries, padded, ground_truth)
    print(
        f"{k_val:>3} | {res.precision:>10.3f} | {res.recall:>8.3f} | {res.f1_score:>6.3f} | {res.mrr_score:>6.3f} | {res.ndcg_score:>6.3f}"
    )

  K |  Precision |   Recall |     F1 |    MRR |   NDCG
-------------------------------------------------------
  1 |      1.000 |    0.389 |  0.560 |  1.000 |  1.000
  2 |      0.833 |    0.667 |  0.741 |  1.000 |  0.871
  3 |      0.667 |    0.778 |  0.718 |  1.000 |  0.823
  5 |      0.467 |    0.889 |  0.612 |  1.000 |  0.890
 10 |      0.233 |    0.889 |  0.370 |  1.000 |  0.890


## 5. Live Recall Measurement

With a live database, use `compute_recall()` to measure ANN vs exact search quality.

```python
# Requires a running pgVectorDB instance:
recall = await db.compute_recall(
    test_queries=["vector search", "database index"],
    k=10,
)
print(f"Recall@10: {recall['recall@k']:.2%}")
```

In [6]:
print("✅ Evaluation complete!")

✅ Evaluation complete!
